In [ ]:
import os
import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

from placer.process import structure

def parse_pdbqt(filepath, format="general"):
    """
    Parse pdbqt file.

    format='general' : single conformer (input ligand), returns coords only
                       output: np.ndarray of shape (n_atoms, 3)
    format='vina'    : multiple conformers (vina output), returns affinity + coords
                       output: list of {"affinity": float, "coords": np.ndarray (n_atoms, 3)}
    """
    conformers = []
    current_coords   = []
    current_affinity = None

    with open(filepath) as f:
        for line in f:
            if line.startswith("MODEL"):
                current_coords   = []
                current_affinity = None
            elif line.startswith("REMARK VINA RESULT"):
                current_affinity = float(line.split()[3])
            elif line.startswith("ATOM") or line.startswith("HETATM"):
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                current_coords.append([x, y, z])
            elif line.startswith("ENDMDL"):
                if current_coords:
                    conformers.append({
                        "affinity": current_affinity,
                        "coords":   np.array(current_coords),
                    })

    # general: no MODEL/ENDMDL blocks
    if not conformers and current_coords:
        conformers.append({
            "affinity": None,
            "coords":   np.array(current_coords),
        })

    if format == "general":
        return conformers[0]["coords"]  # np.ndarray (n_atoms, 3)
    elif format == "vina":
        return conformers               # list of {"affinity", "coords"}
    else:
        raise ValueError(f"Unknown format: '{format}'. Use 'general' or 'vina'.")
    
def get_reference_ligand_coords_from_multimodel(pdb_path, model_idx, ligand_resname="ADI"):
    """
    Description:
        Extract ligand heavy-atom coords from a specific model in a
        multi-model PDB via Bio.PDB.

    Args:
        pdb_path: Path to the multi-model PDB.
        model_idx: 1-based model index.
        ligand_resname: Three-letter residue name of the target ligand.

    Returns:
        np.ndarray of shape (n_atoms, 3); empty array if not found.
    """
    models = structure.load_models_from_pdb(pdb_path)
    model = models[model_idx - 1]   # 1-based -> 0-based
    coords = []
    for res in model.get_residues():
        if res.get_resname().strip() != ligand_resname:
            continue
        for atom in res.get_atoms():
            if atom.element in (None, "H") or atom.get_name().startswith("H"):
                continue
            coords.append(atom.get_coord())
    return np.array(coords)


def select_pose_by_reference(pdbqt_path, ref_coords):
    """
    Description:
        Pick the docking pose closest in heavy-atom RMSD to a reference.

    Args:
        pdbqt_path: Path to vina-generated multi-pose pdbqt file.
        ref_coords: Reference coords (n_atoms, 3), order must match ligand.

    Returns:
        Dict with "affinity", "coords", "rank" (0-based), and "rmsd_to_ref";
        None if no poses or atom count mismatch.
    """
    poses = parse_pdbqt(pdbqt_path, format="vina")
    if not poses or poses[0]["coords"].shape != ref_coords.shape:
        return None
    rmsds = [np.sqrt(((p["coords"] - ref_coords) ** 2).sum() / ref_coords.shape[0])
             for p in poses]
    best = int(np.argmin(rmsds))
    return {
        "affinity": poses[best]["affinity"],
        "coords": poses[best]["coords"],
        "rank": best,
        "rmsd_to_ref": float(rmsds[best]),
    }


def pairwise_rmsd(coords_list):
    """
    Description:
        Mean of pairwise heavy-atom RMSDs between coords, no superposition.

    Args:
        coords_list: List of np.ndarray (n_atoms, 3), all same shape.

    Returns:
        Mean pairwise RMSD; np.nan if fewer than 2 entries.
    """
    n = len(coords_list)
    if n < 2:
        return np.nan
    rmsds = []
    for i in range(n):
        for j in range(i + 1, n):
            diff = coords_list[i] - coords_list[j]
            rmsds.append(np.sqrt((diff ** 2).sum() / diff.shape[0]))
    return float(np.mean(rmsds))


def aggregate_docking_by_reference(docking_dir, ref_root, ligand_resname="ADI",
                                   rmsd_threshold=None):
    """
    Description:
        Aggregate per-entry docking results matched to PLACER multi-model
        reference coords.

    Args:
        docking_dir: Root of docking outputs (entry subfolders).
        ref_root: Root containing multi-model PLACER PDBs
            (one per entry, e.g. carA_<UID>.relax_model.pdb).
        ligand_resname: Ligand resname.
        rmsd_threshold: Optional cutoff for matched-pose RMSD.

    Returns:
        Dict mapping entry to aggregated stats.
    """
    out = {}
    entries = [e for e in sorted(os.listdir(docking_dir)) if e.startswith("carA_")]

    for entry in tqdm(entries, desc="Aggregating"):
        entry_dir = os.path.join(docking_dir, entry)
        ref_pdb = os.path.join(ref_root, f"{entry}.relax_model.pdb")
        if not (os.path.isdir(entry_dir) and os.path.exists(ref_pdb)):
            tqdm.write(f"  [skip] {entry}: missing entry_dir or ref_pdb")
            continue

        per_model = []
        for fname in sorted(os.listdir(entry_dir)):
            if not (fname.startswith("ligand_") and fname.endswith(".pdbqt")):
                continue
            base = fname.replace("ligand_", "").replace(".pdbqt", "")
            model_idx = int(base.split("_")[-1])

            ref_coords = get_reference_ligand_coords_from_multimodel(
                ref_pdb, model_idx, ligand_resname
            )
            if ref_coords.shape[0] == 0:
                continue

            result = select_pose_by_reference(os.path.join(entry_dir, fname), ref_coords)
            if result is None:
                continue
            if rmsd_threshold is not None and result["rmsd_to_ref"] > rmsd_threshold:
                continue
            per_model.append({"model": base, **result})

        if not per_model:
            print(f"  [empty] {entry}: no matched models")
            continue

        n_total = sum(1 for f in os.listdir(entry_dir)
                      if f.startswith("ligand_") and f.endswith(".pdbqt"))

        out[entry] = {
            "affinity_mean": float(np.mean([m["affinity"] for m in per_model])),
            "affinity_std": float(np.std([m["affinity"] for m in per_model])),
            "rank_mean": float(np.mean([m["rank"] for m in per_model])),
            "rmsd_to_ref_mean": float(np.mean([m["rmsd_to_ref"] for m in per_model])),
            "pose_rmsd_mean": pairwise_rmsd([m["coords"] for m in per_model]),
            "n_models": len(per_model),
            "n_total_models": n_total,
            "per_model": per_model,
        }
        print(f"  {entry:<25s} aff={out[entry]['affinity_mean']:>6.2f}  "
              f"rmsd={out[entry]['rmsd_to_ref_mean']:>4.2f}  "
              f"n={out[entry]['n_models']}/{n_total}")

    return out

In [2]:
result_idx = 4

In [3]:
results = aggregate_docking_by_reference(
    docking_dir=f"outputs/docking/carA_homologs_{result_idx}",
    ref_root=f"outputs/placer/carA_holo_adi_amp_homologs_100_{result_idx}",
    ligand_resname="ADI",
    rmsd_threshold=3.0,   # 1 Å 이상 떨어진 pose는 매칭 실패로 간주, 제외
)

Aggregating:   1%|▏         | 1/71 [00:12<14:55, 12.80s/it]

  carA_A0A0F5NA60           aff= -3.80  rmsd=1.72  n=14/15


Aggregating:   3%|▎         | 2/71 [00:32<19:06, 16.62s/it]

  carA_A0A0H3MCY6           aff= -4.64  rmsd=1.69  n=21/22


Aggregating:   4%|▍         | 3/71 [00:51<20:14, 17.86s/it]

...

  carA_H8IU56               aff= -3.80  rmsd=1.60  n=13/14


Aggregating:  99%|█████████▊| 70/71 [17:33<00:15, 15.34s/it]

  carA_K0EY54               aff= -4.64  rmsd=1.51  n=15/19


Aggregating: 100%|██████████| 71/71 [17:47<00:00, 15.03s/it]

  carA_W7J139               aff= -4.55  rmsd=1.97  n=15/15


In [4]:
import requests
from Bio import Entrez, SeqIO
from io import StringIO

Entrez.email = "ghdrms206@gmail.com"


def _fetch_uniprotkb(accession):
    """Return UniProtKB JSON if entry is active and has sequence, else None."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    resp = requests.get(url, timeout=30)
    if resp.status_code != 200:
        return None
    data = resp.json()
    if "sequence" not in data:
        return None
    return data


def _fetch_uniparc_ebi(accession):
    """Return UniParc record from EBI Proteins API (works for obsolete entries)."""
    url = f"https://www.ebi.ac.uk/proteins/api/uniparc/accession/{accession}"
    resp = requests.get(url, headers={"Accept": "application/json"}, timeout=30)
    if resp.status_code != 200:
        return None
    data = resp.json()
    if isinstance(data, list):
        return data[0] if data else None
    return data


def _get_property(xref, key):
    """Helper: extract property value by type from a UniParc dbReference."""
    for p in xref.get("property", []):
        if p.get("type") == key:
            return p.get("value")
    return None


def get_sequence(accession):
    """
    Description:
        Fetch protein sequence; falls back to UniParc (EBI) if UniProtKB lacks it.
    """
    data = _fetch_uniprotkb(accession)
    if data:
        return data["sequence"]["value"]

    archive = _fetch_uniparc_ebi(accession)
    if archive and "sequence" in archive:
        seq = archive["sequence"]
        if isinstance(seq, dict):
            result = seq.get("content") or seq.get("value")
        else:
            result = seq
        if result:
            return result

    print(f"[None] sequence: {accession}")
    return None


def get_taxonomy(accession):
    """
    Description:
        Fetch organism info; falls back to UniParc cross-references if obsolete.
    """
    data = _fetch_uniprotkb(accession)
    if data and "organism" in data:
        org = data["organism"]
        return {
            "accession": accession,
            "tax_id": org["taxonId"],
            "scientific_name": org["scientificName"],
            "common_name": org.get("commonName"),
        }

    archive = _fetch_uniparc_ebi(accession)
    if archive:
        for xref in archive.get("dbReference", []):
            if xref.get("active") != "Y":
                continue
            tax_id = _get_property(xref, "NCBI_taxonomy_id")
            if tax_id:
                return {
                    "accession": accession,
                    "tax_id": int(tax_id),
                    "scientific_name": None,
                    "common_name": None,
                }

    print(f"[None] taxonomy: {accession}")
    return {"accession": accession, "scientific_name": None,
            "tax_id": None, "common_name": None}


def get_dna_from_uniprot(uniprot_accession):
    """
    Description:
        Fetch CDS DNA via EMBL xref; falls back to UniParc (EBI) if obsolete.
    """
    data = _fetch_uniprotkb(uniprot_accession)
    embl_xrefs = []
    if data:
        embl_xrefs = [x for x in data.get("uniProtKBCrossReferences", [])
                      if x["database"] == "EMBL"]

    if not embl_xrefs:
        archive = _fetch_uniparc_ebi(uniprot_accession)
        if archive:
            for xref in archive.get("dbReference", []):
                if xref.get("type") not in ("EMBL", "EMBLWGS"):
                    continue
                if xref.get("active") != "Y":
                    continue
                embl_xrefs.append({
                    "id": xref.get("id"),
                    "properties": [{"key": "ProteinId", "value": xref.get("id")}],
                })
    if not embl_xrefs:
        print(f"[None] dna: {uniprot_accession}")
        return None

    embl_id = embl_xrefs[0]["id"]
    protein_id = None
    for prop in embl_xrefs[0].get("properties", []):
        if prop["key"] == "ProteinId":
            protein_id = prop["value"]
            break

    if protein_id and protein_id != "-":
        handle = Entrez.efetch(db="protein", id=protein_id,
                               rettype="fasta_cds_na", retmode="text")
        fasta_text = handle.read()
        handle.close()
        record = next(SeqIO.parse(StringIO(fasta_text), "fasta"))
        dna_seq = str(record.seq)
    else:
        handle = Entrez.efetch(db="nucleotide", id=embl_id,
                               rettype="fasta", retmode="text")
        record = next(SeqIO.parse(handle, "fasta"))
        handle.close()
        dna_seq = str(record.seq)

    return {"embl_id": embl_id, "protein_id": protein_id, "dna": dna_seq}

In [8]:
df_final = pd.read_csv(f'results/carA_homologs_po_candidates_{result_idx}.txt', sep = '\t')

add = {'affinity_mean': [], 'pose_rmsd_mean': [], 'taxonomy': [], 'sequence': [], 'source_dna': []}
for i, row in tqdm(df_final.iterrows(), total = len(df_final)):
    uniprot_id = row['uniprot_id']

    aff = results['carA_' + uniprot_id]['affinity_mean']
    rmsd = results['carA_' + uniprot_id]['pose_rmsd_mean']
    tax = get_taxonomy(uniprot_id)
    seq = get_sequence(uniprot_id)
    dna = get_dna_from_uniprot(uniprot_id)

    add['affinity_mean'].append(aff)
    add['pose_rmsd_mean'].append(rmsd)
    add['taxonomy'].append(tax['scientific_name'])
    add['sequence'].append(seq)
    add['source_dna'].append(dna['dna'])
    

df_final = df_final.assign(**add)
df_final = df_final.sort_values(by = 'affinity_mean')
df_final

100%|██████████| 71/71 [08:20<00:00,  7.05s/it]


,uniprot_id,nac_fraction_holo,nac_holo_idxs,n_confident_models,prmsd_mean,prmsd_std,nac_fraction_apo,nac_apo_idxs,affinity_mean,pose_rmsd_mean,taxonomy,sequence,source_dna
32,A0A3S4RS92,0.395833,2;4;5;6;9;10;12;14;18;20;22;25;30;32;38;41;43;...,48,2.414383,1.088687,0.72,2;3;4;5;6;7;8;9;12;13;14;15;16;17;18;21;22;23;...,-4.662882,2.369113,Mycolicibacterium aurum,MSTATREERLESRIAELFATDHQFAEAAPDAAITDAIDAAGSRLPQ...,ATGTCGACTGCTACCCGCGAGGAGCGGCTCGAGAGCCGCATCGCCG...
69,K0EY54,0.431818,2;4;5;7;9;14;17;21;22;24;26;28;29;30;31;32;33;...,44,2.450372,1.040333,0.94,1;2;3;4;5;6;7;8;9;10;11;12;13;14;15;16;17;18;1...,-4.643200,1.702329,Nocardia brasiliensis (strain ATCC 700358 / HU...,MFAEDEQVKAAVPDQEVVEAIRAPGLRLAQIMATVMERYADRPAVG...,TTGTTCGCCGAGGACGAGCAGGTGAAAGCCGCGGTGCCGGACCAGG...
1,A0A0H3MCY6,0.392857,1;2;5;9;10;11;14;18;19;20;21;25;26;27;29;35;38...,56,2.246222,1.320387,0.89,1;2;3;4;5;6;7;9;10;11;12;13;16;17;18;20;21;22;...,-4.637667,2.099672,None,MSINDQRLTRRVEDLYASDAQFAAASPNEAITQAIDQPGVALPQLI...,ATGTCGATCAACGATCAGCGACTGACACGCCGCGTCGAGGACCTAT...
62,A0AAU4K3W1,0.568627,1;2;3;4;5;7;9;11;14;19;21;23;25;26;30;31;33;34...,51,2.461432,1.183558,0.96,1;2;3;4;5;6;7;8;9;10;11;12;13;14;15;16;17;18;1...,-4.573345,2.335226,Williamsia herbipolensis,MSTPTQDDTADTPSRDELDAHVAERLRRLTENDPQVAAALPNPDLS...,ATGAGCACACCCACCCAGGACGACACCGCCGACACCCCGAGCCGCG...
65,A0AB72XQA2,0.312500,3;6;8;11;14;19;21;23;30;31,32,2.718294,1.178290,0.70,3;4;5;6;7;8;9;11;14;15;16;17;19;20;22;24;26;27...,-4.550900,2.177982,None,MSINDQRLTRRVEDLYASDAQFAAASPNEAITQAIDQPGVALPQLI...,ATGTCGATCAACGATCAGCGACTGACACGCCGCGTCGAGGACCTAT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,A0A179V396,0.348837,2;4;8;13;17;24;26;29;32;34;36;37;41;42;43,43,2.620467,1.058389,0.64,1;2;3;5;6;7;9;11;13;15;16;17;19;21;22;24;25;27...,-3.552846,2.585820,None,MTVTNETDLQQEQLSLRVEKLRETDQQFRSALPDPEVTKQVLRPGL...,ATGACTGTGACCAATGAAACCGACCTGCAGCAGGAGCAGCTGTCCC...
50,A0A7X6MIZ0,0.365854,1;2;3;5;6;7;8;12;13;16;21;31;33;35;40,41,2.611690,1.211802,0.85,1;2;3;4;6;7;8;9;11;12;13;14;16;17;18;19;20;21;...,-3.515133,2.202306,None,MTTETREDRLQRRIATLYETDSQFADARPSDAVNAAVAQPELRLPA...,ATGACCACCGAAACACGCGAGGACCGGCTCCAACGCCGGATCGCAA...
24,A0A1X0XWA4,0.333333,1;3;7;11;14;18;22;24;26;28;32,33,2.818753,1.050055,0.57,1;2;4;5;7;8;9;11;12;13;15;16;17;18;19;20;21;23...,-3.495091,2.198271,Mycobacterium simiae,MSTISEERLARRVEELTASDPQFAAARPDPAIVEALEQPGLRLPQI...,ATGTCTACTATTTCCGAAGAACGGCTAGCCCGCAGAGTCGAGGAAC...
7,A0A178LTI6,0.441176,8;9;12;14;17;18;20;25;27;28;29;30;31;32;34,34,2.971108,1.168482,0.74,1;2;3;4;5;7;8;9;10;11;12;13;14;15;16;17;18;19;...,-3.424500,3.603528,None,MSTREDRLERRIAQLFSTDCQFAQAAPDAGIAAAIDAPDLTLPSIV...,ATGTCGACTCGCGAGGATCGCCTCGAGCGCCGCATCGCCCAATTGT...


In [9]:
df_final.to_csv(f'results/20260607_carA_homologs_po_ds_candidates_{result_idx}.csv', index = False)

In [ ]:

from Bio.Seq import Seq

for i, row in df_final.iterrows():
    uniprot_id = row["uniprot_id"]
    aa_seq = row["sequence"]
    dna_seq = row["source_dna"]
    
    dna2aa = Seq(dna_seq).translate()
    print(f"{uniprot_id}: {aa_seq == (str(dna2aa)[:-1])}")

A0A3S4RS92: True
K0EY54: False
A0A0H3MCY6: True
A0AAU4K3W1: True
E5XP76: True
A0AA37PJ36: True
A0A6G9XT36: True
A0A1R3Y1N0: True
A0A1W9YPH5: True
A0A502EC16: True
A0A1A2DP38: True
A0A846XPH2: True
A0A318K9K9: True
A0A064CG00: True
A0A1E3SV84: True
V5XIA1: False
A0A1V3WG34: True
A0A1D8GAR9: True
A0A0U1E1C0: True
A0A1X1U567: True
A0A7Z0IKF1: True
A0A7I7Q331: True
A0A1B8SKL4: True
A0A7I7UBW3: True
A0A934NT38: True
A0A1X1WD57: True
A0AAI8U017: True
A0A7K3LE40: True
A0A927MND6: True
A0A7I9XMW5: True
A0A1A2SKN5: True
A0A7I7X9S2: True
A0AAC9YL18: True
A0A4Z0HRY8: True
A0A6G3SLA6: True
A0A1Y5PCF7: True
H8IU56: True
A0AB73U9A3: True
A0A1E3RBW0: True
A0A0F4ES51: True
A0A498PZU2: True
A0AA37PRM7: True
A0A8E2LPD0: True
A0A829Q1V2: True
A0AB38USB2: True
A0A829MDQ7: True
A0A7I7JNU6: True
A0A0I9Z3I8: True
A0AB38CZ04: True
A0A179V396: True
A0A1S1L7L3: True
A0A7V8RXZ3: True
A0A0U0ZG49: True
A0AAD1I1H5: True
A0A1X1ZAJ3: True
A0A178LTI6: True
F5YUX6: True
A0A1A2EVY2: True
O69484: True
